# FlowGuard++ Attack Suite Experiments

This notebook is a compact control panel for the FlowGuard++ evaluation runbook:

- run the new attack grid (`A1`, `A2`, `A3`, `Adaptive-Adaptive`, plus `MAZE` baseline)
- load the generated suite summaries and Markdown report
- visualize AUROC, FPR, detection rate, and substitute accuracy across defenses
- inspect query images from generated `visualization_records.pt` / `transferset.pickle` artifacts
- run or visualize the post-hoc `A4` Sparse-Probe and `A5` LabelMaze analyses when their inputs exist

The default configuration is smoke-sized. Increase the query budgets after the wiring looks good.

## Assumptions and Trade-offs

Assumptions:
- A trained victim checkpoint exists at `runs/notebook/training-victim-cifar10-vgg16_bn-nodefense/target_model`.
- A deployed FlowPure/FlowGuard CNF exists at `runs/flow_matching/cifar10_flowpure_pgd/checkpoint_latest.pt`.
- The notebook kernel is the repository `.venv`, or at least has the same dependencies installed.

Trade-offs:
- The suite is launched through the same scripts as the runbook, keeping experiment logic out of the notebook.
- `RUN_INLINE_GRID` and `RUN_POSTHOC` are `False` by default so opening the notebook never starts long GPU jobs accidentally.
- Visualizations degrade gracefully when only some attack/defense pairs have finished.

In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

try:
    import torch
except ImportError:  # The summary visualizations do not require torch.
    torch = None

project_root = Path.cwd().resolve()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

NATURE = {
    "paper": "#F4F1E8",
    "ink": "#2E2A26",
    "moss": "#607D3B",
    "forest": "#355E3B",
    "clay": "#B5654A",
    "bark": "#6D4C41",
    "sky": "#5C7FA3",
    "sand": "#D8C8A8",
    "stone": "#7D8A91",
}


def apply_nature_style() -> None:
    plt.rcParams.update(
        {
            "figure.facecolor": NATURE["paper"],
            "axes.facecolor": NATURE["paper"],
            "savefig.facecolor": NATURE["paper"],
            "axes.edgecolor": NATURE["ink"],
            "axes.labelcolor": NATURE["ink"],
            "text.color": NATURE["ink"],
            "xtick.color": NATURE["ink"],
            "ytick.color": NATURE["ink"],
            "grid.color": "#C7BCA9",
            "grid.alpha": 0.35,
            "axes.grid": True,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "font.family": "DejaVu Serif",
            "figure.dpi": 150,
        }
    )


apply_nature_style()
print(f"Project root: {project_root}")

In [ ]:
RUN_ROOT = project_root / "runs" / "notebook" / "flowguard_suite_2048"
TARGET_CHECKPOINT_DIR = (
    project_root
    / "runs"
    / "notebook"
    / "training-victim-cifar10-vgg16_bn-nodefense"
    / "target_model"
)
FLOW_CHECKPOINT = (
    project_root / "runs" / "flow_matching" / "cifar10_flowpure_pgd" / "checkpoint_latest.pt"
)
A3_SURROGATE_CHECKPOINT = FLOW_CHECKPOINT
PYTHON = project_root / ".venv" / "Scripts" / "python.exe"
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

DEFENSES = ["flowpure", "flowguard_integral", "flowguard_userlevel"]
ATTACKS = [
    "a1_clean_transfer",
    "a2_projected_maze",
    "a3_flowblind",
    "adaptive_adaptive",
    "maze_baseline",
]

RUN_INLINE_GRID = False
RUN_POSTHOC = False
FORCE_RERUN = False

DEVICE = "cuda"
DATASET = "CIFAR10"
QUERY_DATASET = "CIFAR10"
CLEAN_TRANSFER_DATASET = "CIFAR100"
TARGET_FPR = 0.05
ATTACK_QUERY_BUDGET = 144
BENIGN_QUERY_BUDGET = 144
ATTACK_BATCH_SIZE = 1
EXTRACTION_ATTACK_BATCH_SIZE = 16
TRAINING_BATCH_SIZE = 16
EPOCHS = 1
SEED_SAMPLES = 16

print(f"Python: {PYTHON}")
print(f"Output root: {RUN_ROOT}")

In [ ]:
def assert_preflight() -> None:
    missing = [
        path
        for path in [TARGET_CHECKPOINT_DIR, FLOW_CHECKPOINT, A3_SURROGATE_CHECKPOINT]
        if not path.exists()
    ]
    if missing:
        lines = "\n".join(f"- {path}" for path in missing)
        raise FileNotFoundError(
            "Missing required artifact(s). Update the config cell before running experiments:\n"
            f"{lines}"
        )


def format_command(command: list[str]) -> str:
    return " ".join(shlex.quote(str(part)) for part in command)


def build_suite_command() -> list[str]:
    command = [
        str(PYTHON),
        str(project_root / "scripts" / "evaluate_flowguard_suite.py"),
        "--output-root",
        str(RUN_ROOT),
        "--defenses",
        ",".join(DEFENSES),
        "--attacks",
        ",".join(ATTACKS),
        "--dataset",
        DATASET,
        "--query-dataset",
        QUERY_DATASET,
        "--clean-transfer-dataset",
        CLEAN_TRANSFER_DATASET,
        "--target-checkpoint-dir",
        str(TARGET_CHECKPOINT_DIR),
        "--flow-checkpoint",
        str(FLOW_CHECKPOINT),
        "--a3-surrogate-checkpoint",
        str(A3_SURROGATE_CHECKPOINT),
        "--device",
        DEVICE,
        "--target-fpr",
        str(TARGET_FPR),
        "--attack-query-budget",
        str(ATTACK_QUERY_BUDGET),
        "--benign-query-budget",
        str(BENIGN_QUERY_BUDGET),
        "--attack-batch-size",
        str(ATTACK_BATCH_SIZE),
        "--extraction-attack-batch-size",
        str(EXTRACTION_ATTACK_BATCH_SIZE),
        "--training-batch-size",
        str(TRAINING_BATCH_SIZE),
        "--epochs",
        str(EPOCHS),
        "--seed-samples",
        str(SEED_SAMPLES),
    ]
    if FORCE_RERUN:
        command.append("--force")
    return command


assert_preflight()
suite_command = build_suite_command()
print(format_command(suite_command))

In [ ]:
if RUN_INLINE_GRID:
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(suite_command, cwd=project_root, check=True)
else:
    print("RUN_INLINE_GRID=False, so no experiments were started.")
    print("Set RUN_INLINE_GRID=True in the config cell to launch the grid.")

In [ ]:
def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def load_suite_rows(run_root: Path) -> list[dict[str, Any]]:
    summary_path = run_root / "flowguard_suite_summary.json"
    if summary_path.exists():
        payload = load_json(summary_path)
        return [dict(row) for row in payload.get("results", [])]

    rows: list[dict[str, Any]] = []
    for metrics_path in sorted(run_root.glob("*/attack_suite_metrics.json")):
        pair_name = metrics_path.parent.name
        if "__" not in pair_name:
            continue
        defense, _attack = pair_name.split("__", 1)
        payload = load_json(metrics_path)
        for row in payload.get("results", []):
            merged = dict(row)
            merged["defense"] = defense
            rows.append(merged)
    return rows


rows = load_suite_rows(RUN_ROOT)
if not rows:
    raise FileNotFoundError(
        f"No FlowGuard++ suite results found below {RUN_ROOT}. Run the grid cell first."
    )

defenses = sorted({str(row.get("defense", "unknown")) for row in rows})
attacks = sorted({str(row.get("attack", "unknown")) for row in rows})
print(f"Loaded {len(rows)} result rows")
print(f"Defenses: {defenses}")
print(f"Attacks: {attacks}")
rows[:3]

In [ ]:
report_path = RUN_ROOT / "flowguard_suite_report.md"
if report_path.exists():
    display(Markdown(report_path.read_text(encoding="utf-8")))
else:
    print(f"Markdown report not found: {report_path}")

In [ ]:
def to_float(value: Any) -> float:
    if value is None:
        return float("nan")
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")


def metric_matrix(metric: str) -> tuple[np.ndarray, list[str], list[str]]:
    defenses_local = sorted({str(row.get("defense", "unknown")) for row in rows})
    attacks_local = sorted({str(row.get("attack", "unknown")) for row in rows})
    lookup = {
        (str(row.get("defense", "unknown")), str(row.get("attack", "unknown"))): row
        for row in rows
    }
    matrix = np.full((len(attacks_local), len(defenses_local)), np.nan, dtype=np.float64)
    for i, attack in enumerate(attacks_local):
        for j, defense in enumerate(defenses_local):
            row = lookup.get((defense, attack), {})
            matrix[i, j] = to_float(row.get(metric))
    return matrix, attacks_local, defenses_local


def plot_metric_heatmap(metric: str, title: str, vmin: float = 0.0, vmax: float = 1.0) -> None:
    matrix, attack_labels, defense_labels = metric_matrix(metric)
    fig, ax = plt.subplots(figsize=(1.8 * len(defense_labels) + 2.5, 0.55 * len(attack_labels) + 2.0))
    image = ax.imshow(matrix, cmap="YlGnBu", vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_title(title)
    ax.set_xticks(np.arange(len(defense_labels)), labels=defense_labels, rotation=25, ha="right")
    ax.set_yticks(np.arange(len(attack_labels)), labels=attack_labels)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            label = "n/a" if np.isnan(value) else f"{value:.3f}"
            ax.text(j, i, label, ha="center", va="center", color=NATURE["ink"], fontsize=9)
    fig.colorbar(image, ax=ax, shrink=0.8)
    fig.tight_layout()
    plt.show()


plot_metric_heatmap("auroc", "AUROC by defense and attack")
plot_metric_heatmap("fpr", "Observed FPR by defense and attack")
plot_metric_heatmap("detection_rate", "Attack detection rate by defense and attack")

In [ ]:
substitute_rows = [
    row
    for row in rows
    if row.get("substitute_accuracy") is not None and not row.get("error")
]

if substitute_rows:
    labels = [f"{row['attack']}\n{row['defense']}" for row in substitute_rows]
    values = [to_float(row.get("substitute_accuracy")) for row in substitute_rows]
    fig, ax = plt.subplots(figsize=(max(8, 0.55 * len(labels)), 4))
    ax.bar(np.arange(len(values)), values, color=NATURE["moss"])
    ax.set_title("Substitute accuracy reported by attack artifacts")
    ax.set_ylabel("accuracy")
    ax.set_xticks(np.arange(len(labels)), labels=labels, rotation=45, ha="right")
    fig.tight_layout()
    plt.show()
else:
    print("No substitute_accuracy values found in the loaded suite rows.")

In [ ]:
def load_torch_payload(path: Path) -> Any:
    if torch is None:
        raise ImportError("torch is required for loading visualization records")
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def find_attack_artifact(defense: str, attack: str, filename: str) -> Path | None:
    pair_dir = RUN_ROOT / f"{defense}__{attack}"
    if not pair_dir.exists():
        return None
    matches = sorted(pair_dir.rglob(filename))
    return matches[0] if matches else None


def query_tensor_from_record(record: Any) -> Any:
    if isinstance(record, dict):
        value = record.get("query_x", record.get("x"))
    elif isinstance(record, (tuple, list)) and record:
        value = record[0]
    else:
        raise TypeError(f"Unsupported visualization record type: {type(record)!r}")
    return value if torch is None else torch.as_tensor(value)


def to_cifar_image_array(value: Any) -> np.ndarray:
    tensor = torch.as_tensor(value).detach().cpu().float() if torch is not None else np.asarray(value)
    if torch is not None:
        if tensor.ndim == 4:
            tensor = tensor[0]
        if tensor.ndim != 3:
            raise ValueError(f"Expected CHW image tensor, got shape {tuple(tensor.shape)}")
        if float(tensor.min()) < -0.1 or float(tensor.max()) > 1.1:
            mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
            std = torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1)
            tensor = tensor * std + mean
        array = tensor.clamp(0, 1).permute(1, 2, 0).numpy()
    else:
        array = np.asarray(tensor)
        if array.shape[0] == 3:
            array = np.transpose(array, (1, 2, 0))
        array = np.clip(array, 0.0, 1.0)
    return array


def show_query_grid(defense: str, attack: str, count: int = 12) -> None:
    artifact = find_attack_artifact(defense, attack, "visualization_records.pt")
    if artifact is None:
        artifact = find_attack_artifact(defense, attack, "transferset.pickle")
    if artifact is None:
        print(f"No visualization artifact found for {defense} x {attack}")
        return

    records = load_torch_payload(artifact)
    if not records:
        print(f"Artifact is empty: {artifact}")
        return

    chosen = list(records[: min(count, len(records))])
    columns = min(6, len(chosen))
    rows_count = int(np.ceil(len(chosen) / columns))
    fig, axes = plt.subplots(rows_count, columns, figsize=(1.8 * columns, 1.9 * rows_count))
    axes_array = np.asarray(axes).reshape(-1)
    for ax, record in zip(axes_array, chosen):
        image = to_cifar_image_array(query_tensor_from_record(record))
        ax.imshow(image)
        ax.set_axis_off()
    for ax in axes_array[len(chosen) :]:
        ax.set_axis_off()
    fig.suptitle(f"{defense} x {attack}: query samples")
    fig.tight_layout()
    plt.show()
    print(f"Loaded: {artifact}")


show_query_grid("flowpure", "a2_projected_maze", count=12)

In [ ]:
def pair_config(defense: str, attack: str) -> dict[str, Any]:
    metrics_path = RUN_ROOT / f"{defense}__{attack}" / "attack_suite_metrics.json"
    if not metrics_path.exists():
        return {}
    return dict(load_json(metrics_path).get("config", {}))


def first_artifact(defense: str, attack: str, filename: str) -> Path | None:
    return find_attack_artifact(defense, attack, filename)


A4_BENIGN_SCORES = RUN_ROOT / "a4_inputs" / "benign_scores.json"
A4_ATTACK_SCORES = RUN_ROOT / "a4_inputs" / "a3_flowblind_scores.json"
A4_OUTPUT = RUN_ROOT / "a4_sparse_probe_report.json"
A5_OUTPUT = RUN_ROOT / "a5_labelmaze_report.json"

threshold = pair_config("flowpure", "a1_clean_transfer").get("calibrated_threshold")
a5_transferset = first_artifact("flowpure", "a1_clean_transfer", "transferset.pickle")

if RUN_POSTHOC:
    if A4_BENIGN_SCORES.exists() and A4_ATTACK_SCORES.exists() and threshold is not None:
        sparse_command = [
            str(PYTHON),
            str(project_root / "scripts" / "analyze_sparse_probe.py"),
            "--benign-scores",
            str(A4_BENIGN_SCORES),
            "--attack-scores",
            str(A4_ATTACK_SCORES),
            "--threshold",
            str(threshold),
            "--output-json",
            str(A4_OUTPUT),
        ]
        print(format_command(sparse_command))
        subprocess.run(sparse_command, cwd=project_root, check=True)
    else:
        print("Skipping A4: score JSON inputs or calibrated threshold are missing.")

    if a5_transferset is not None and threshold is not None:
        labelmaze_command = [
            str(PYTHON),
            str(project_root / "scripts" / "run_labelmaze_attack.py"),
            "--transferset",
            str(a5_transferset),
            "--flow-checkpoint",
            str(FLOW_CHECKPOINT),
            "--threshold",
            str(threshold),
            "--select-k",
            "1024",
            "--output-json",
            str(A5_OUTPUT),
        ]
        print(format_command(labelmaze_command))
        subprocess.run(labelmaze_command, cwd=project_root, check=True)
    else:
        print("Skipping A5: A1 transferset or calibrated threshold is missing.")
else:
    print("RUN_POSTHOC=False, so A4/A5 scripts were not started.")
    print(f"A5 transferset candidate: {a5_transferset}")
    print(f"Calibrated threshold candidate: {threshold}")

In [ ]:
if A4_OUTPUT.exists():
    sparse_payload = load_json(A4_OUTPUT)
    sparse_rows = sparse_payload.get("results", [])
    fig, ax = plt.subplots(figsize=(7, 4))
    for window in sorted({int(row["user_window"]) for row in sparse_rows}):
        subset = [row for row in sparse_rows if int(row["user_window"]) == window]
        subset = sorted(subset, key=lambda row: float(row["probe_rate"]))
        ax.plot(
            [float(row["probe_rate"]) for row in subset],
            [float(row["user_level_auroc"]) for row in subset],
            marker="o",
            label=f"window={window}",
        )
    ax.set_title("A4 Sparse-Probe user-level AUROC")
    ax.set_xlabel("probe rate")
    ax.set_ylabel("user-level AUROC")
    ax.set_ylim(0.45, 1.02)
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print(f"A4 report not found: {A4_OUTPUT}")

if A5_OUTPUT.exists():
    labelmaze_payload = load_json(A5_OUTPUT)
    stats = labelmaze_payload.get("stats", [])
    labels = [str(row["subset"]) for row in stats]
    rates = [float(row["detection_rate"]) for row in stats]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(labels, rates, color=[NATURE["moss"], NATURE["sky"], NATURE["clay"]][: len(labels)])
    ax.axhline(TARGET_FPR, color=NATURE["bark"], linestyle="--", label=f"target FPR={TARGET_FPR:.2f}")
    ax.set_title("A5 LabelMaze FlowPure detection rate")
    ax.set_ylabel("detection rate")
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print(f"A5 report not found: {A5_OUTPUT}")